# `load.ipynb` - Carga idempotente en PostgreSQL (UPSERT) + cuarentena

Requisito: *"La inserción en la base de datos destino debe utilizar lógica de Upsert (ON CONFLICT DO UPDATE) para permitir re-ejecuciones sin duplicar registros."*

- `tb_indicadores_europa`: `INSERT ... ON CONFLICT (nombre_pais) DO UPDATE SET ...` - al re-ejecutar el pipeline, los 44 países se **actualizan** en lugar de duplicarse (la columna `nombre_pais` es `UNIQUE`).
- `tb_cuarentena_geodatos`: inserción en bloque (`execute_values`) de todos los registros rechazados, enlazados al `id_ejecucion` actual para trazabilidad.
- Toda la carga ocurre dentro de una única transacción: si algo falla, se hace `ROLLBACK` y se relanza la excepción.

> Depende de `config.ipynb` y `db.ipynb` (usa `conectar_postgres`). **No incluye una celda de prueba aislada** a propósito: escribe en las tablas destino, así que la forma correcta de probarla es la ejecución real del pipeline completo en `main.ipynb` (sección 6), no una llamada suelta aquí que dejaría datos de prueba mezclados con los reales.

In [ ]:
import logging

import pandas as pd
import psycopg2.extras

## Función principal

In [ ]:
def cargar_en_postgres(df_indicadores: pd.DataFrame, df_cuarentena: pd.DataFrame, id_ejecucion: int):
    """Carga idempotente (UPSERT) de indicadores + insercion de cuarentena."""
    conn = conectar_postgres()
    cur = conn.cursor()
    insertados = 0
    try:
        sql_upsert = """
            INSERT INTO tb_indicadores_europa
                (nombre_pais, poblacion_total, superficie_km2, pib_total_eur, densidad_poblacional,
                 pib_per_capita_eur, fuente_poblacion, fecha_actualizacion)
            VALUES (%s, %s, %s, %s, %s, %s, %s, NOW())
            ON CONFLICT (nombre_pais) DO UPDATE SET
                poblacion_total = EXCLUDED.poblacion_total,
                superficie_km2 = EXCLUDED.superficie_km2,
                pib_total_eur = EXCLUDED.pib_total_eur,
                densidad_poblacional = EXCLUDED.densidad_poblacional,
                pib_per_capita_eur = EXCLUDED.pib_per_capita_eur,
                fuente_poblacion = EXCLUDED.fuente_poblacion,
                fecha_actualizacion = NOW();
        """
        for fila in df_indicadores.itertuples(index=False):
            cur.execute(sql_upsert, (
                fila.nombre_pais, int(fila.poblacion_total), float(fila.superficie_km2),
                float(fila.pib_total_eur), float(fila.densidad_poblacional),
                float(fila.pib_per_capita_eur), fila.fuente_poblacion,
            ))
            insertados += 1
        logger.info(f"UPSERT completado en tb_indicadores_europa: {insertados} paises.")

        if not df_cuarentena.empty:
            sql_cuarentena = """
                INSERT INTO tb_cuarentena_geodatos
                    (pais_original, superficie_valor_orig, superficie_unidad_orig,
                     pib_valor_orig, pib_divisa_orig, motivo_rechazo, id_ejecucion)
                VALUES %s;
            """
            valores = [
                (
                    r.pais_original,
                    None if pd.isna(r.superficie_valor_orig) else float(r.superficie_valor_orig),
                    r.superficie_unidad_orig,
                    None if pd.isna(r.pib_valor_orig) else float(r.pib_valor_orig),
                    r.pib_divisa_orig,
                    r.motivo_rechazo,
                    id_ejecucion,
                )
                for r in df_cuarentena.itertuples(index=False)
            ]
            psycopg2.extras.execute_values(cur, sql_cuarentena, valores)
            logger.info(f"Insertados {len(valores)} registros en tb_cuarentena_geodatos.")

        conn.commit()
    except Exception:
        conn.rollback()
        logger.error("Error durante la carga en PostgreSQL. Se hace ROLLBACK de la transaccion.", exc_info=True)
        raise
    finally:
        cur.close()
        conn.close()
    return insertados